In [5]:
from Bio import AlignIO, SeqIO
from Bio.SeqRecord import SeqRecord
import numpy as np

def find_insertions(domains_file, alignment_file, output_file, min_len=30, gap_thr=0.6):
    domains = {}
    with open(domains_file, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            p = line.strip().split()
            if len(p) < 23:
                continue
            try:
                sid = p[3]
                start = int(p[19])
                end = int(p[20])
            except:
                continue
            if sid not in domains:
                domains[sid] = []
            domains[sid].append((start, end))

    for sid in domains:
        domains[sid].sort()

    aln = AlignIO.read(alignment_file, "fasta")
    arr = np.array([list(r.seq) for r in aln])

    found = []
    for i, rec in enumerate(aln):
        sid = rec.id
        if sid not in domains:
            continue

        L = arr.shape[1]
        pos = 0

        while pos < L:
            if arr[i, pos] != "-":
                other = np.concatenate([arr[:i, pos], arr[i+1:, pos]])
                if np.sum(other == "-") / len(other) >= gap_thr:
                    s = pos
                    while pos < L and arr[i, pos] != "-":
                        other2 = np.concatenate([arr[:i, pos], arr[i+1:, pos]])
                        if np.sum(other2 == "-") / len(other2) < gap_thr:
                            break
                        pos += 1
                    e = pos

                    seq = "".join(arr[i, s:e])
                    real_len = len(seq)

                    if real_len >= min_len and real_len % 3 == 0:

                        ok = True
                        for ds, de in domains[sid]:
                            if not (e < ds or s > de):
                                ok = False
                                break

                        if ok:
                            found.append((sid, seq, s+1, e))
                else:
                    pos += 1
            else:
                pos += 1

    records = []
    for n, (sid, seq, s, e) in enumerate(found, 1):
        records.append(SeqRecord(
            seq=seq,
            id=f"{sid} {s}-{e}",
            description=f"len={len(seq)}"
        ))

    SeqIO.write(records, output_file, "fasta")

    print(len(found))
    return found


find_insertions(
    domains_file=r"C:\Users\2slon\OneDrive\Рабочий стол\5sem\pestiviruses\final\domresults.tbl",
    alignment_file=r"C:\Users\2slon\OneDrive\Рабочий стол\5sem\pestiviruses\final\f_pal_mafft.fasta",
    output_file=r"C:\Users\2slon\OneDrive\Рабочий стол\5sem\pestiviruses\final\unannotated_insertions.fasta",
    min_len=30,
    gap_thr=0.6
)


242


[('PX236819.1', 'CTGGGTTGTCTGCAAGAGCCTAAAATCTCCAGT', 700, 732),
 ('PX236819.1', 'GAAGGAGAAGAAGAAGAGGAAGTAACAACCTCTGAA', 23731, 23766),
 ('PX236820.1',
  'CTGAACCTATACAGTGTGATAGCACAACGGCTGGGTTGTCTGCAAGAGCCTAAAGTCTCCAGT',
  670,
  732),
 ('PX236820.1', 'GAAGGAGAAGAAGGAGAAGAAGCAACAATCTCTGAA', 23731, 23766),
 ('PX236821.1', 'CTGGGTTGTCTGCGAGAGCCTGAAGTCTCCAGT', 700, 732),
 ('PX236821.1', 'GAAGGAGAAGAAGAAGAAGAAGTAACAATCCCTGAA', 23731, 23766),
 ('PX236822.1', 'CTGGGTTGTCTGCAAGAGCCTAAAGTCTCCAGT', 700, 732),
 ('PX236822.1', 'GAAGGAGAAGAAGAAGAAGAAGTAACAATCCCTGAA', 23731, 23766),
 ('PX275358.1',
  'ATGTGCAGCCGATGCCAGGGAAAGCATAGGAGGTTTGAAATGGACCGGGAACCTAAGAGTGCCAGATACTGTGCTGAGTGTAATAGGCTGCATCCTGCTGAGGAAGGTGACTTTTGGGCAGAGTCGAGCATGTTGGGCCTCAAAATCACCTACTTTGCGCTGATGGATGGAAAGGTGTATGATATCACAGAGTGGGCTGGATGCCAGCGTGTGGGAATCTCCCCAGATACCCACAGAGTCCCTTGTCACATCTCATTTGGTTCACGGATG',
  10522,
  10791),
 ('LC860282.1', 'GAAGAGGAAGACGAGAGTGAAGAGTTGATCACT', 23731, 23763),
 ('PV626377.1',
  'AAGCATAGGAGGTTTGAAGTGGACCG